# 02.01 · Extracción de texto de los IDOs

Se extraen de los IDOs en Word de `data/raw/2. IDOs - Input CCI/` las ocho secciones que alimentan
la tabla de registro Input CCI y el insumo de Evidencia + Recomendaciones (paso 3):

| Plantilla del IDO | Se usa |
|---|---|
| FICHA TÉCNICA DE LA OPERACIÓN | ✔ |
| ACCIÓN A SOMETER A CONSIDERACIÓN | |
| 1. INFORMACIÓN DEL CLIENTE | |
| 2. DETALLES DE LA OPERACIÓN | |
| 2.1. ANTECEDENTES | ✔ |
| 2.2. DESCRIPCIÓN | ✔ |
| 2.3. INFORMACIÓN PRELIMINAR DEL NEGOCIO POTENCIAL | ✔ |
| 3. JUSTIFICACIÓN Y BENEFICIOS ESPERADOS | ✔ |
| 4. RIESGOS Y OTROS PUNTOS CRÍTICOS ASOCIADOS A LA OPERACIÓN (4.1 Riesgos) | ✔ |
| 5. MARCO AMBIENTAL Y SOCIAL · 5.1. ANÁLISIS PRELIMINAR DE RIESGOS E IMPACTOS AMBIENTALES Y SOCIALES | ✔ (solo 5.1) |
| 5.2. ANÁLISIS PRELIMINAR DE FINANCIAMIENTO VERDE | ✔ |
| 5.3. ANÁLISIS PRELIMINAR DE ALINEACIÓN CON EL ACUERDO DE PARÍS, 5.4 RECOMENDACIONES PARA FORTALECER LOS CO-BENEFICIOS AMBIENTALES, CLIMÁTICOS Y DE BIODIVERSIDAD | |
| 6. EQUIPO DE TRABAJO PROPUESTO | |
| 7. RECOMENDACIONES | |
| 8. OPINIÓN DE LA GCR | |

Todos los Word se leen en cada ejecución. El proceso es **local**: no se usa la API y tarda
pocos segundos. El resultado es una sola tabla, `data/interim/2. IDOs - Input CCI/02.01_extraccion_idos/idos.csv`, con
una fila por IDO; es la entrada del notebook `02.02`.

## 1. LOAD LIBRARIES

Se usan la biblioteca estándar y pandas. Se ubica la carpeta del proyecto (la que tiene
`pyproject.toml`) para que el notebook funcione abierto desde `notebooks/` o desde la raíz, y
las rutas se toman de `caf_input_cci_preparatoria/config.py`.

In [1]:
from datetime import datetime
from decimal import ROUND_HALF_UP, Decimal
import hashlib
import html
from pathlib import Path
import re
import sys
import zipfile

from IPython.display import HTML, display
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Se toman todos los `.docx` de la carpeta, salvo los temporales de Word (`~$...`). El texto de
cada Word se lee directamente de su XML interno, con un salto de línea por párrafo.

Además del texto, de cada archivo se guardan dos datos:

| Dato | Qué es |
|---|---|
| `huella` | SHA-256 del archivo. Cambia si cambia el contenido, aunque el nombre sea el mismo. Los notebooks siguientes la usan para no volver a llamar a Azure si el Word no cambió. |
| `fecha_registro` | Día en que el Word se creó en la carpeta (fecha del sistema de archivos, no la que Word guarda por dentro). |

Si dos archivos tienen la misma huella, son el mismo documento: se conserva el primero en orden
alfabético y se omite el otro.

In [2]:
# Rutas
carpeta_idos = config.RAW_IDOS_INPUT
archivo_salida = config.INTERIM / "2. IDOs - Input CCI" / "02.01_extraccion_idos" / "idos.csv"

rutas = sorted(p for p in carpeta_idos.glob("*.docx") if not p.name.startswith("~$"))
assert rutas, f"No hay archivos .docx en {carpeta_idos}"

In [3]:
# Función para extraer el texto de XML (Extensible Markup Language) del documento word
def texto_docx(ruta):
    xml = zipfile.ZipFile(ruta).read("word/document.xml").decode("utf-8")
    xml = xml.replace("</w:p>", "\n").replace("<w:tab/>", " ")
    return html.unescape(re.sub(r"<[^>]+>", "", xml)).replace("\ufeff", "")

In [4]:
# Arma la tabla de partida: una fila por cada Word de la carpeta, quita los repetidos y muestra la lista
documentos = pd.DataFrame(
    [
        {
            "archivo": ruta.name,
            "huella": hashlib.sha256(ruta.read_bytes()).hexdigest(),
            "fecha_registro": datetime.fromtimestamp(ruta.stat().st_birthtime).date().isoformat(),
            "texto": texto_docx(ruta),
        }
        for ruta in rutas
    ]
)
duplicados = documentos["huella"].duplicated()
for archivo in documentos.loc[duplicados, "archivo"]:
    print(f"Se omite {archivo}: es idéntico a otro Word de la carpeta.")
documentos = documentos.loc[~duplicados].reset_index(drop=True)
documentos.index += 1  # numeración desde 1

print(f"{len(documentos)} IDOs en {carpeta_idos.relative_to(ROOT)}:")
display(documentos[["archivo", "fecha_registro"]])

15 IDOs en data/raw/IDOs Input CCI:


,archivo,fecha_registro
1,10 09 2026 - IDO PAED Barbados Programa Seguri...,2026-10-12
2,10 14 2026 - Borrador1 IDO ICE FR 228.docx,2026-10-21
3,10 19 2026 - FR-228 - ANDE 8.docx,2026-10-19
4,10 21 2026 - IDO Proyecto San Jacinto.docx,2026-10-21
5,1errefref - CCI IDO 1806 PB GM FO l.docx,2026-10-12
6,BORRADOR CCI IDO_Programa OSE4-0.docx,2026-10-26
7,FR 228 El Salvador Vuela II 21 9 2026 comparti...,2026-11-04
8,FR-228 IDO Bergantin_RD.docx,2026-11-04
9,FR-228 guanay actualizado.docx,2026-11-04
10,IDO Cierre de Brechas Chancay V18Ago2026 copy....,2026-10-12


## 3. PROCESS DATA

### Las ocho secciones

De la plantilla del IDO se usan solo estas secciones:

| Sección | Clave |
|---|---|
| FICHA TÉCNICA | `ficha_tecnica` |
| 2.1 ANTECEDENTES | `2.1_antecedentes` |
| 2.2 DESCRIPCIÓN | `2.2_descripcion` |
| 2.3 INFORMACIÓN PRELIMINAR DEL NEGOCIO POTENCIAL | `2.3_negocio_potencial` |
| 3 JUSTIFICACIÓN Y BENEFICIOS ESPERADOS | `3_justificacion_beneficios` |
| 4 RIESGOS Y OTROS PUNTOS CRÍTICOS (4.1 Riesgos) | `4.1_riesgos` |
| 5.1 ANÁLISIS PRELIMINAR DE RIESGOS E IMPACTOS AMBIENTALES Y SOCIALES | `5.1_riesgos_ambientales_sociales` |
| 5.2 ANÁLISIS PRELIMINAR DE FINANCIAMIENTO VERDE | `5.2_financiamiento_verde` |

Cada sección va desde su título hasta el título siguiente de la plantilla, y se guarda todo el
texto que hay entre ambos, tal cual. Un título se reconoce cuando ocupa la línea completa, con o
sin su número y sin distinguir mayúsculas. Si falta el título de inicio o el de cierre, la
sección queda vacía y aparece en el control de calidad.

Las secciones 3, 4.1 y 5.1 no se usan en el Input CCI: se extraen para el insumo de Evidencia +
Recomendaciones del paso 3. La sección 4 de la plantilla solo tiene la 4.1, así que se corta
desde el título de la sección 4 hasta MARCO AMBIENTAL Y SOCIAL. Las secciones se buscan en el
orden del Word, por eso el diccionario sigue ese orden.

In [5]:
# Definir la numeración posible de las secciones
NUMERO = r"(\d+(\.\d+)*\.?\s*)?"  # numeración opcional: "2.1.", "5.2 ", "3. "

# Mapea los cortes de las secciones, dónde inicia y dónde termina una sección (de las 8 que interesan)
SECCIONES = {  # sección: (título donde empieza, título siguiente de la plantilla)
    "ficha_tecnica": (
        r"FICHA TÉCNICA DE LA OPERACIÓN",
        r"ACCIÓN A SOMETER A CONSIDERACIÓN",
    ),
    "2.1_antecedentes": (
        r"ANTECEDENTES",
        r"DESCRIPCIÓN",
    ),
    "2.2_descripcion": (
        r"DESCRIPCIÓN",
        r"INFORMACIÓN PRELIMINAR DEL NEGOCIO POTENCIAL",
    ),
    "2.3_negocio_potencial": (
        r"INFORMACIÓN PRELIMINAR DEL NEGOCIO POTENCIAL",
        r"JUSTIFICACIÓN Y BENEFICIOS ESPERADOS",
    ),
    "3_justificacion_beneficios": (
        r"JUSTIFICACIÓN Y BENEFICIOS ESPERADOS",
        r"RIESGOS Y OTROS PUNTOS CRÍTICOS ASOCIADOS A LA OPERACIÓN",
    ),
    "4.1_riesgos": (
        r"RIESGOS Y OTROS PUNTOS CRÍTICOS ASOCIADOS A LA OPERACIÓN",
        r"MARCO AMBIENTAL Y SOCIAL",
    ),
    "5.1_riesgos_ambientales_sociales": (
        r"ANÁLISIS PRELIMINAR DE RIESGOS E IMPACTOS AMBIENTALES Y SOCIALES",
        r"(ANÁLISIS PRELIMINAR DE )?FINANCIAMIENTO VERDE",
    ),
    "5.2_financiamiento_verde": (
        r"(ANÁLISIS PRELIMINAR DE )?FINANCIAMIENTO VERDE",
        r"ANÁLISIS PRELIMINAR DE ALINEACIÓN CON EL ACUERDO DE PARÍS"
        r"|OPORTUNIDADES DE CO-BENEFICIOS.*|RECOMENDACIONES PARA FORTALECER.*"
        r"|EQUIPO DE TRABAJO PROPUESTO",
    ),
}


# Se divide el texto en líneas (una por párrafo), se eliminan las líneas vacías, los espacios repetidos o tabulaciones se reducen a un solo espacio
def lineas_limpias(texto):
    return [re.sub(r"\s+", " ", linea).strip() for linea in texto.splitlines() if linea.strip()]


# se encuentra un título en una de las líneas en las que fue organizado el texto (paso anterior), se devuelve el número de la primera línea que es exactamente el título
def buscar_titulo(lineas, titulo, desde):
    patron = re.compile(rf"{NUMERO}({titulo})\s*:?", re.IGNORECASE)
    return next((i for i in range(desde, len(lineas)) if patron.fullmatch(lineas[i])), None)


# Se cortan las ocho secciones, que busca las lineas que estan entre el texto del título de la sección y la sección en la que termina
def extraer_secciones(texto):
    """{sección: texto entre su título y el siguiente}; vacía si falta alguno de los dos."""
    lineas, desde, secciones = lineas_limpias(texto), 0, {}
    for clave, (inicio, siguiente) in SECCIONES.items():
        a = buscar_titulo(lineas, inicio, desde)
        b = buscar_titulo(lineas, siguiente, a + 1) if a is not None else None
        secciones[clave] = "\n".join(lineas[a + 1 : b]) if b is not None else ""
        desde = a + 1 if a is not None else desde
    return secciones

### Datos de la ficha técnica

La ficha alterna etiqueta y valor (`Operación`, `País`, `Cliente`...). Se toma el texto entre
una etiqueta y la siguiente, sin distinguir mayúsculas. En los campos con casillas
(*Riesgo institucional*, *Modalidad*) se conserva la opción marcada con ☒.
Los montos se obtienen exclusivamente de `2.3_negocio_potencial`.

In [6]:
def entre(texto, desde, hasta):
    m = re.search(rf"(?:^|\n){desde}\b(.*?)\n(?:{hasta})", texto, re.DOTALL | re.IGNORECASE)
    return re.sub(r"\s+", " ", m[1]).strip(" .") or None if m else None


def marcado(valor):
    m = re.search(r"☒\s*([^☐☒]+)", valor or "")
    return m[1].strip(" .:") if m else valor


# Función para extraer datos de la sección Ficha técnica
def datos_ficha(ficha):
    return {
        "operacion": entre(ficha, r"Operaci[óo]n", r"Pa[íi]s"),
        "pais": entre(ficha, r"Pa[íi]s", r"Cliente"),
        "cliente": entre(ficha, r"Cliente", r"Garante|Organismo"),
        "riesgo": marcado(entre(ficha, r"Riesgo institucional", r"Modalidad")),
        "modalidad": marcado(entre(ficha, r"Modalidad", r"Monto")),
    }

### Agendas misionales y transversales

Se guarda en una sola columna, `agendas`, el contenido de la ficha entre la línea
`Agendas misionales y transversales` y la siguiente que empieza por `Nivel de desarrollo de`.
Se excluyen ambos títulos y se conservan los subtítulos, el texto y los saltos de línea.
Si falta alguno de los límites, el cuadro queda vacío para revisarlo.

El control de calidad avisa **agenda sin completar en el IDO** si el cuadro no contiene un
código como `A3`, `B1a` o `A.3`. El aviso no altera el contenido ni crea otra columna.

In [7]:
CODIGO_AGENDA = r"\b[ABC]\.?\d+[a-z]?\b"


# Función para extraer datos de la sección Ficha técnica - Agendas
def extraer_agendas(ficha):
    """Conserva las líneas del cuadro de agendas, sin los títulos que lo delimitan."""
    lineas = ficha.splitlines()
    inicio = next(
        (
            i + 1
            for i, linea in enumerate(lineas)
            if linea.strip().casefold() == "agendas misionales y transversales"
        ),
        None,
    )
    if inicio is None:
        return ""
    fin = next(
        (
            i
            for i in range(inicio, len(lineas))
            if lineas[i].lstrip().casefold().startswith("nivel de desarrollo de")
        ),
        None,
    )
    return "\n".join(lineas[inicio:fin]) if fin is not None else ""

### Tres montos de 2.3, en millones de USD

| Columna | Etiqueta de origen |
|---|---|
| `monto_total` | Monto total |
| `a_local_prestamos` | Capital / Aporte local |
| `monto_caf` | Financiamiento directo de CAF, con o sin `(USD)` |

Cada bloque termina en la siguiente etiqueta. `Préstamos` es una etiqueta delimitadora.
Se guardan solo estas tres columnas numéricas, redondeadas a dos decimales. El `monto_caf`
de la ficha técnica se sustituye por el financiamiento directo de CAF de 2.3.

Punto significa miles y coma significa decimales; `MM` y `millones` ya expresan millones.
`$` equivale a USD. Se ignoran `Por hasta`, `Hasta`, el cierre `.-` y los porcentajes entre
paréntesis. `-`, `NA`, `N/A` y `No aplica` quedan vacíos sin aviso.

Primero se aplica la regla directa. Solo si falla, se buscan cifras precedidas de `USD`, `US$`
o `$` en todo el bloque, con o sin paréntesis y con espacios dentro de la cifra (por ejemplo,
`USD 198.000. 000`). Los porcentajes se ignoran. Si todos los importes encontrados equivalen al
mismo valor, se convierte a millones con las mismas reglas y se redondea al final.

Si hay varios importes distintos, o ninguno porque el monto solo está escrito en palabras,
el valor queda vacío y el control de calidad muestra el bloque para revisarlo. El texto
original se conserva en la columna `2.3_negocio_potencial`.

In [8]:
ETIQUETAS_MONTOS = {
    "monto_total": r"Monto\s+total",
    "a_local_prestamos": r"Capital\s*/\s*Aporte\s+local",
    "monto_caf": r"Financiamiento\s+directo\s+de\s+CAF(?:\s*\(USD\))?",
}
COLUMNAS_MONTOS = list(ETIQUETAS_MONTOS)
OTRAS_ETIQUETAS = (
    r"Pr[eé]stamos|Cofinanciamiento\s*/\s*financiamiento\s+catal[ií]tico"
    r"|Otras\s+fuentes\s+de\s+financiamiento(?:\s*\([^)]*\))?"
)
PATRON_ETIQUETAS = re.compile(
    r"^[ \t]*(?:"
    + "|".join(f"(?P<{campo}>{etiqueta})" for campo, etiqueta in ETIQUETAS_MONTOS.items())
    + "|"
    + OTRAS_ETIQUETAS
    + r")[ \t]*:?[ \t]*$",
    re.IGNORECASE | re.MULTILINE,
)


# Función para extraer datos de la sección 2.3 Información preliminar del negocio potencial
def bloques_montos(texto):
    """Conserva el bloque de cada monto hasta la siguiente etiqueta de financiamiento."""
    etiquetas = list(PATRON_ETIQUETAS.finditer(texto))
    bloques = dict.fromkeys(COLUMNAS_MONTOS, "")
    for i, etiqueta in enumerate(etiquetas):
        campo = etiqueta.lastgroup
        if campo and not bloques[campo]:
            fin = etiquetas[i + 1].start() if i + 1 < len(etiquetas) else len(texto)
            bloques[campo] = texto[etiqueta.end() : fin].strip()
    return bloques

In [9]:
def no_aplica_monto(texto):
    return not texto.strip() or bool(
        re.fullmatch(r"(?:[-–—]|NA|N/A|No\s+aplica)[.\s]*", texto.strip(), flags=re.IGNORECASE)
    )


def importe_directo_en_millones(texto):
    """Aplica la regla original y devuelve millones sin redondear."""
    if no_aplica_monto(texto):
        return None
    limpio = re.sub(r"\([^)]*%[^)]*\)", "", texto)
    limpio = re.sub(r"\s+", " ", limpio).strip()
    limpio = re.sub(r"^(?:Por\s+)?Hasta\s+", "", limpio, flags=re.IGNORECASE)
    limpio = re.sub(r"\.\s*-$", "", limpio).strip()
    coincidencia = re.match(
        r"^(?:(?:USD|US\s*\$|\$)\s*)?"
        r"(?P<numero>\d+(?:\.\s*\d{3})*(?:,\d{1,2})?)"
        r"(?P<unidad>\s*(?:MM|millones|mill[oó]n)\b)?(?P<resto>.*)$",
        limpio,
        flags=re.IGNORECASE,
    )
    if not coincidencia:
        return None
    resto = coincidencia["resto"]
    if re.search(r"[\d%]|\b(?:mil|miles|billones|bill[oó]n)\b", resto, flags=re.IGNORECASE):
        return None
    numero = re.sub(r"\s+", "", coincidencia["numero"]).replace(".", "").replace(",", ".")
    monto = Decimal(numero)
    if not coincidencia["unidad"]:
        monto /= Decimal(1_000_000)
    return monto

In [10]:
def monto_en_millones(texto):
    monto = importe_directo_en_millones(texto)
    if monto is None and not no_aplica_monto(texto):
        coincidencias = re.finditer(
            r"(?<!\w)(?:USD|US\s*\$|\$)\s*"
            r"(?P<numero>\d(?:[\d.,\s]*\d)?)"
            r"(?P<unidad>\s*(?:MM|millones|mill[oó]n|miles|mil|billones|bill[oó]n)\b|\s*%)?",
            texto,
            flags=re.IGNORECASE,
        )
        importes = set()
        for coincidencia in coincidencias:
            unidad = coincidencia["unidad"] or ""
            if unidad.strip() == "%":
                continue
            numero = re.sub(r"\s+", "", coincidencia["numero"])
            importe = importe_directo_en_millones(numero + unidad)
            if importe is None:
                return None
            importes.add(importe)
        if len(importes) == 1:
            monto = importes.pop()
    if monto is None:
        return None
    return float(monto.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


def montos_seccion(texto):
    return {campo: monto_en_millones(bloque) for campo, bloque in bloques_montos(texto).items()}

### Financiamiento verde de 5.2

Se lee únicamente `5.2_financiamiento_verde` y se calculan tres columnas:

- `fin_verde_marcado`: la única opción marcada con ☒ entre Elegible, Potencialmente elegible
  y No elegible. `☐☒` cuenta como marcada. Las casillas de Ambiente y Biodiversidad, Mitigación
  y Adaptación se ignoran. Si no hay una opción única, el valor queda vacío.
- `fin_verde_sugerido`: se examinan las frases posteriores al título `Financiamiento Verde
  potencial`, incluido `Criterio técnico`, y se descartan las instrucciones de la plantilla.
  Las reglas se aplican en este orden: No elegible (`no es elegible`, `no son elegibles`,
  `no resulta elegible`); Potencialmente elegible (`potencialmente elegible`, `elegibilidad
  potencial`, `financiamiento verde potencial de hasta X%`); Elegible (`es elegible`,
  `son elegibles`, `resulta elegible`). Sin estas frases, el valor queda vacío.
- `fin_verde_frase`: la primera frase que cumple la regla de mayor prioridad, para revisarla
  junto a la opción marcada.

El control de calidad señala las discrepancias cuando ambos valores existen y son distintos.
`02.05` toma la opción marcada para completar el Excel.

In [11]:
COLUMNAS_FIN_VERDE = ["fin_verde_marcado", "fin_verde_sugerido", "fin_verde_frase"]


def financiamiento_verde_marcado(texto):
    opciones = re.findall(
        r"☒\s*(Potencialmente\s+elegible|No\s+elegible|Elegible)\b",
        texto,
        flags=re.IGNORECASE,
    )
    catalogo = {
        opcion.casefold(): opcion
        for opcion in ("Elegible", "Potencialmente elegible", "No elegible")
    }
    marcadas = {catalogo[re.sub(r"\s+", " ", opcion).casefold()] for opcion in opciones}
    return marcadas.pop() if len(marcadas) == 1 else ""

In [12]:
REGLAS_FIN_VERDE = [
    ("No elegible", r"\b(?:no\s+es\s+elegible|no\s+son\s+elegibles|no\s+resulta\s+elegible)\b"),
    (
        "Potencialmente elegible",
        (
            r"\b(?:potencialmente\s+elegible\b|elegibilidad\s+potencial\b|"
            r"financiamiento\s+verde\s+potencial\s+de\s+hasta\s+\d+(?:[.,]\d+)?\s*%)"
        ),
    ),
    ("Elegible", r"\b(?:es\s+elegible|son\s+elegibles|resulta\s+elegible)\b"),
]


def sugerir_financiamiento_verde(texto):
    inicio = re.search(
        r"(?:^|\n)[ \t]*Financiamiento\s+Verde\s+potencial\b[ \t]*:?[ \t]*",
        texto,
        flags=re.IGNORECASE,
    )
    if inicio is None:
        return "", ""
    bloques = re.split(
        r"(?:^|\n)[ \t]*Criterio\s+t[eé]cnico\b[ \t]*:?[ \t]*",
        texto[inicio.end() :],
        flags=re.IGNORECASE,
    )
    frases = [
        frase.strip()
        for bloque in bloques
        for frase in re.split(r"(?<=[.!?])\s+", bloque)
        if frase.strip()
        and not re.match(
            r"^\s*(?:[-•]\s*)?(?:Señalar|Indicar|Describir|Especificar|Completar)\b",
            frase,
            flags=re.IGNORECASE,
        )
    ]
    for opcion, patron in REGLAS_FIN_VERDE:
        for frase in frases:
            if re.search(patron, frase, flags=re.IGNORECASE):
                return opcion, frase
    return "", ""

### Tabla por IDO

Se reúnen en una fila por IDO la huella, la fecha de registro, los datos de la ficha, las
agendas, los tres montos, el financiamiento verde y el texto completo de las ocho secciones.
Las secciones se guardan tal cual, porque `02.02` envía al modelo las de la ficha y la 2 para
redactar los textos, y el paso 3 usa las 3, 4.1 y 5.1 para el insumo de Evidencia +
Recomendaciones.

In [13]:
filas = []
for documento in documentos.itertuples():
    secciones = extraer_secciones(documento.texto)
    ficha = secciones["ficha_tecnica"]
    verde = secciones["5.2_financiamiento_verde"]
    fin_verde_sugerido, fin_verde_frase = sugerir_financiamiento_verde(verde)
    filas.append(
        {
            "archivo": documento.archivo,
            "huella": documento.huella,
            "fecha_registro": documento.fecha_registro,
            **datos_ficha(ficha),
            "agendas": extraer_agendas(ficha),
            **montos_seccion(secciones["2.3_negocio_potencial"]),
            "fin_verde_marcado": financiamiento_verde_marcado(verde),
            "fin_verde_sugerido": fin_verde_sugerido,
            "fin_verde_frase": fin_verde_frase,
            **secciones,
        }
    )

idos = pd.DataFrame(filas, index=documentos.index)
idos[COLUMNAS_MONTOS] = idos[COLUMNAS_MONTOS].astype("Float64")
print(f"{len(idos)} IDOs procesados")

15 IDOs procesados


### Control de calidad

Se muestran la ficha, los tres montos de 2.3 en **millones de USD** con dos decimales, el
financiamiento verde y el número de caracteres de cada sección, que permite detectar las
secciones vacías. La columna `agendas` conserva sus saltos de línea en la vista.

Debajo se listan tres avisos: los IDOs cuya opción de financiamiento verde marcada y sugerida
son distintas, los cuadros de agendas vacíos o sin código y los bloques de montos con texto no
convertible. Los valores no aplicables no generan avisos.

In [14]:
columnas_ficha = ["archivo", "fecha_registro", "pais", "riesgo", "modalidad", "agendas"]
fin_verde_discrepante = (
    idos["fin_verde_marcado"].ne("")
    & idos["fin_verde_sugerido"].ne("")
    & idos["fin_verde_marcado"].ne(idos["fin_verde_sugerido"])
)
vista_calidad = idos[columnas_ficha + COLUMNAS_MONTOS + COLUMNAS_FIN_VERDE].copy()
vista_calidad[list(SECCIONES)] = idos[list(SECCIONES)].map(len)
vista_calidad["Revisar financiamiento verde"] = fin_verde_discrepante.map(
    {True: "Marcado y sugerido distintos", False: ""}
)
vista_calidad = vista_calidad.map(
    lambda valor: html.escape(valor) if isinstance(valor, str) else valor
)
vista_calidad["agendas"] = vista_calidad["agendas"].str.replace("\n", "<br>", regex=False)
with pd.option_context("display.max_colwidth", None):
    display(HTML(vista_calidad.to_html(escape=False, na_rep="", float_format="{:.2f}".format)))

vacias = idos[list(SECCIONES)].eq("").sum()
print("Secciones vacías:", vacias[vacias > 0].to_dict() or "ninguna")

,archivo,fecha_registro,pais,riesgo,modalidad,agendas,monto_total,a_local_prestamos,monto_caf,fin_verde_marcado,fin_verde_sugerido,fin_verde_frase,ficha_tecnica,2.1_antecedentes,2.2_descripcion,2.3_negocio_potencial,5.2_financiamiento_verde,Revisar financiamiento verde
1,10 09 2026 - IDO PAED Barbados Programa Seguridad y Justicia.docx,2026-10-12,Barbados,Soberano,Préstamo de Libre Disponibilidad – Apoyo a Estrategias de Desarrollo (PAED),A3. Territorios resilientes.B3. Fortalecimiento institucional.,75.00,,75.00,,,,3744,7188,2555,2184,384,
2,10 14 2026 - Borrador1 IDO ICE FR 228.docx,2026-10-21,República de Costa Rica,Cuasi-Soberano,Línea de crédito (no soberano),"Completar según el apartado 3.1, Agendas misionales y transversales",100.00,,100.00,,,,3887,8391,9183,1702,384,
3,10 19 2026 - FR-228 - ANDE 8.docx,2026-10-19,Paraguay,Soberano,Programa/proyecto de inversión (PPI),"A1. Transición energética justaB1. Operación más verde y B4. Inclusión, diversidad y género",263.00,,263.00,Potencialmente elegible,,,5095,40699,19944,2160,786,
4,10 21 2026 - IDO Proyecto San Jacinto.docx,2026-10-21,Bolivia,Soberano,Programa/proyecto de inversión (PPI),Agenda Misional:A3a. Fomentar la seguridad hídricaAgendas transversales:B1a. Implementar operaciones más verdes.,122.25,23.46,98.79,Elegible,,,5278,1951,4667,2407,2031,
5,1errefref - CCI IDO 1806 PB GM FO l.docx,2026-10-12,Ecuador,Soberano,Programa/proyecto de inversión (PPI),A.3 Territorios resilientes B1 Una operación más verde C6 Instrumentos financieros innovadores,90.00,15.00,75.00,,,,3579,8768,4696,1567,384,
6,BORRADOR CCI IDO_Programa OSE4-0.docx,2026-10-26,Uruguay,Soberano,Programa/proyecto de inversión (PPI),A3. Territorios ResilientesA3a. Fomentar la Seguridad HídricaA5. Infraestructura física y digitalB1. Una Operación más verde,63.00,,63.00,Elegible,Elegible,La operación es elegible a Financiamiento Verde (FV).,4168,4527,7109,2282,1469,
7,FR 228 El Salvador Vuela II 21 9 2026 compartido.docx,2026-11-04,El Salvador,Soberano,Programa/proyecto de inversión (PPI),A5. Infraestructura Física y Digital/ A5b. Promover la conectividad física y la transformación digital.B5. Integración regional/ B5A. Impulsar proyectos de integración multinacionales y binacionales.,,,,Potencialmente elegible,Potencialmente elegible,La operación es potencialmente elegible a Financiamiento Verde.,5522,6995,7602,2760,852,
8,FR-228 IDO Bergantin_RD.docx,2026-11-04,República Dominicana,Cuasi-Soberano,Financiamiento estructurado,A5. Infraestructura física y digital.Transversal: B6. Alianzas y movilización de recursos.,,,,Potencialmente elegible,,,3406,1972,8288,3554,1630,
9,FR-228 guanay actualizado.docx,2026-11-04,Estado Plurinacional de Bolivia,Soberano,Programa/proyecto de inversión (PPI),Agenda Misional:A5. Infraestructura física y digitalAgendas transversales:B5. Integración Regional –Impulsar proyectos de integración multinacionales y binacionales.,43.78,0.00,43.78,Elegible,,,4090,2508,1885,1984,1930,
10,IDO Cierre de Brechas Chancay V18Ago2026 copy.docx,2026-10-12,República del Perú,Soberano,Programa/proyecto de inversión (PPI),"Agenda Misional:A3. Territorios ResilientesA3a. Fomentar la seguridad hídricaA3b. Desarrollar sistemas urbanosAgenda Transversal:B1. Una Operación Más VerdeB1a. Implementar operaciones más verdesB3. Fortalecimiento InstitucionalB3a. Fortalecer los gobiernos regionales y localesB4. Inclusión, Diversidad y GéneroB4a. Transversalizar la perspectiva de género, inclusión y diversidad.",198.00,49.00,198.00,,Potencialmente elegible,"Se estima preliminarmente un financiamiento verde potencial de hasta 100% del monto CAF, sujeto a confirmación durante la evaluación técnica, ambiental, social y financiera.",5280,12896,10681,3106,2008,


Secciones vacías: ninguna


In [15]:
discrepancias_fin_verde = idos.loc[fin_verde_discrepante, ["archivo", *COLUMNAS_FIN_VERDE]]
if discrepancias_fin_verde.empty:
    print("Financiamiento verde: sin discrepancias entre la opción marcada y la sugerida.")
else:
    print(f"Financiamiento verde: revisar discrepancias en {len(discrepancias_fin_verde)} IDOs.")
    display(HTML(discrepancias_fin_verde.to_html(index=False)))

Financiamiento verde: revisar discrepancias en 1 IDOs.


archivo,fin_verde_marcado,fin_verde_sugerido,fin_verde_frase
IDO Programa Vial 26-06-26.docx,Elegible,Potencialmente elegible,"La operación es potencialmente elegible de financiamiento verde, principalmente en el objetivo de adaptación al cambio climático."


In [16]:
archivos_sin_agenda = idos.loc[
    ~idos["agendas"].str.contains(CODIGO_AGENDA, case=False, na=False), "archivo"
]
for archivo in archivos_sin_agenda:
    print(f"{archivo}: agenda sin completar en el IDO")
if archivos_sin_agenda.empty:
    print("Todos los cuadros contienen al menos un código de agenda.")

10 14 2026 - Borrador1 IDO ICE FR 228.docx: agenda sin completar en el IDO


In [17]:
montos_por_revisar = pd.DataFrame(
    [
        {"archivo": archivo, "campo": campo, "texto": texto}
        for archivo, seccion in zip(idos["archivo"], idos["2.3_negocio_potencial"])
        for campo, texto in bloques_montos(seccion).items()
        if not no_aplica_monto(texto) and monto_en_millones(texto) is None
    ],
    columns=["archivo", "campo", "texto"],
)
if montos_por_revisar.empty:
    print("Todos los montos con importe numérico se pudieron convertir.")
else:
    print(f"Montos por revisar en {montos_por_revisar['archivo'].nunique()} IDOs:")
    montos_por_revisar.index += 1
    with pd.option_context("display.max_colwidth", None):
        display(montos_por_revisar)

Montos por revisar en 2 IDOs:


,archivo,campo,texto
1,FR 228 El Salvador Vuela II 21 9 2026 compartido.docx,monto_total,"Por hasta USD 60,000,000.00"
2,FR 228 El Salvador Vuela II 21 9 2026 compartido.docx,a_local_prestamos,USD 0.00
3,FR 228 El Salvador Vuela II 21 9 2026 compartido.docx,monto_caf,"USD 60,000,000.00"
4,FR-228 IDO Bergantin_RD.docx,monto_total,"Macroinfraestructura: aproximadamente USD 269 MM según actualización del cliente y USD 276,1 MM según la matriz de costes, pendiente de conciliación. Componente hotelero: presupuesto total pendiente de definición para el hotel o los hoteles que recibirán hasta USD 15 MM de CAF."
5,FR-228 IDO Bergantin_RD.docx,a_local_prestamos,"Macroinfraestructura: aportes y derechos fiduciarios reportados por el cliente, sujetos a conciliación con estados auditados y cuentas del proyecto. Hoteles: aportes de Punta Bergantín y de los socios privados de cada vehículo, por definir y verificar durante la evaluación."
6,FR-228 IDO Bergantin_RD.docx,monto_caf,Hasta USD 75 MM: hasta USD 60 MM para macroinfraestructura y hasta USD 15 MM para componentes hoteleros.


### Revisar un IDO

Para leer el texto extraído y compararlo con el Word, se cambian `ido_a_revisar` (número de
fila de la tabla de control de calidad) y `seccion_a_revisar`: "ficha_tecnica"; "2.1_antecedentes"; "2.2_descripcion"; "2.3_negocio_potencial"; "3_justificacion_beneficios"; "4.1_riesgos";
"5.1_riesgos_ambientales_sociales"; "5.2_financiamiento_verde"


In [18]:
ido_a_revisar = 7
seccion_a_revisar = "ficha_tecnica"

texto = idos.loc[ido_a_revisar, seccion_a_revisar]
print(idos.loc[ido_a_revisar, "archivo"], "·", seccion_a_revisar, f"({len(texto)} caracteres)\n")
print(texto[:3000])

FR 228 El Salvador Vuela II 21 9 2026 compartido.docx · ficha_tecnica (5522 caracteres)

Operación
Programa El Salvador Vuela – Fase II.
País
El Salvador.
Cliente
República de El Salvador.
Garante
N.A.
Organismo ejecutor
Comisión Ejecutiva Portuaria Autónoma (CEPA).
Sector institucional
☒ Público
☐ Privado
☐ Mixto / Indicar % de participación privada
Riesgo institucional
☒ Soberano
☐ Cuasi-Soberano
☐ No Soberano
Modalidad
☒ Programa/proyecto de inversión (PPI)
☐ Préstamo Financiero Soberano
☐ Préstamo por Resultados (PPR)
☐ Préstamo de Libre Disponibilidad – Basado en Matriz de Políticas Públicas (PBL)
☐ Préstamo de Libre Disponibilidad – Apoyo a Estrategias de Desarrollo (PAED)
☐ Préstamo de Libre Disponibilidad – Liquidez de mediano o largo plazo (PLIQ)
☐ Préstamo de Libre Disponibilidad – Liquidez de corto plazo (FEL)
☐ Línea de Crédito Contingente para Eventos Extremos del Clima, Sismos, Accidentes Contaminantes y Epidemias
☐ Línea de Crédito Contingente de Liquidez
☐ Línea de Créd

## 4. SAVE DATA

Se guarda `data/interim/2. IDOs - Input CCI/02.01_extraccion_idos/idos.csv` (la carpeta se crea si no existe). Se
reescribe completo en cada ejecución, con una fila por IDO y estas columnas:

| Columnas | Contenido |
|---|---|
| `archivo`, `huella`, `fecha_registro` | Identificación del Word |
| `operacion`, `pais`, `cliente`, `riesgo`, `modalidad`, `agendas` | Datos de la ficha técnica |
| `monto_total`, `a_local_prestamos`, `monto_caf` | Montos de 2.3, en millones de USD |
| `fin_verde_marcado`, `fin_verde_sugerido`, `fin_verde_frase` | Financiamiento verde de 5.2 |
| `ficha_tecnica`, `2.1_antecedentes`, `2.2_descripcion`, `2.3_negocio_potencial`, `3_justificacion_beneficios`, `4.1_riesgos`, `5.1_riesgos_ambientales_sociales`, `5.2_financiamiento_verde` | Texto completo de las ocho secciones |

Los Word de `data/raw/` no se modifican.

In [19]:
archivo_salida.parent.mkdir(parents=True, exist_ok=True)
idos.to_csv(archivo_salida, index=False, encoding="utf-8-sig", float_format="%.2f")
print(f"{len(idos)} IDOs guardados en {archivo_salida.relative_to(ROOT)}")

15 IDOs guardados en data/interim/01.01_extraccion_idos/idos.csv
